# Benchmark de clasificación
Compara EfficientNet-B0, ResNet50 y ViT-B/16 con el mismo split.


In [ ]:
!pip install -q -U scikit-learn pandas


In [ ]:
from copy import deepcopy
from pathlib import Path
import json
import time

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

DATA_ROOT = Path("/kaggle/input/tu-dataset-clasificacion")
EPOCHS = 5
BATCH = 32
LR = 3e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder(DATA_ROOT / "train", transform=transform)
val_ds = datasets.ImageFolder(DATA_ROOT / "val", transform=transform)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
NUM_CLASSES = len(train_ds.classes)
print(DEVICE, train_ds.classes)


In [ ]:
def build_model(name):
    if name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    elif name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    elif name == "vit_b_16":
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        model.heads.head = nn.Linear(model.heads.head.in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return model.to(DEVICE)

def train_one(model):
    optimizer = AdamW(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    best_state = None
    best_f1 = -1.0
    start = time.perf_counter()
    for _ in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        model.eval()
        truth, pred = [], []
        with torch.inference_mode():
            for images, labels in val_loader:
                logits = model(images.to(DEVICE))
                pred.extend(logits.argmax(1).cpu().tolist())
                truth.extend(labels.tolist())
        score = f1_score(truth, pred, average="macro")
        if score > best_f1:
            best_f1 = score
            best_state = deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    truth, pred = [], []
    with torch.inference_mode():
        for images, labels in val_loader:
            logits = model(images.to(DEVICE))
            pred.extend(logits.argmax(1).cpu().tolist())
            truth.extend(labels.tolist())
    return {
        "accuracy": accuracy_score(truth, pred),
        "macro_f1": f1_score(truth, pred, average="macro"),
        "seconds": time.perf_counter() - start,
    }


In [ ]:
rows = []
states = {}
for name in ["efficientnet_b0", "resnet50", "vit_b_16"]:
    model = build_model(name)
    metrics = train_one(model)
    states[name] = deepcopy(model.state_dict())
    rows.append({"model": name, **metrics})

results = pd.DataFrame(rows).sort_values("macro_f1", ascending=False).reset_index(drop=True)
results


In [ ]:
best_name = results.iloc[0]["model"]
torch.save(states[best_name], f"/kaggle/working/{best_name}_best.pth")
Path("/kaggle/working/classes.json").write_text(json.dumps(train_ds.classes, ensure_ascii=False), encoding="utf-8")
results.to_csv("/kaggle/working/classification_benchmark.csv", index=False)
print(best_name)
